# 🌍 Climate Zone Data Analysis
### Exploring How Temperature Varies with Latitude

---

**Learning Objectives:**
By the end of this lab you will be able to:
- Load and inspect real-world weather station data using pandas
- Calculate annual mean temperatures from monthly values
- Create and interpret scatter plots of temperature vs. latitude
- Group data into latitude bins and compute group averages
- Discuss how spatial averaging affects data visualization

---

**Instructions:** This notebook has **fill-in-the-blank** sections marked with `???`. Replace each `???` with the correct code before running that cell. Answer the discussion questions in the Markdown cells provided.

---
## Part 0 — Import Libraries

Run the cell below. You do not need to change anything here — it loads the Python libraries we will use throughout the lab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Make plots look a bit nicer
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully!')

---
## Part 1 — Load and Explore the Data

The dataset `Monthly_Temperature_Update2.xls` contains mean monthly surface temperatures (°C) for over 500 weather stations worldwide.  
Each row is one station. The columns are:

| Column | Description |
|--------|-------------|
| `Station` | Station ID |
| `Lat` / `Lon` | Latitude/Longitude (text, e.g. `78.1N`) |
| `Lat_num` / `Lon_num` | Latitude/Longitude as numbers (positive = N/E) |
| `Elev` | Elevation (m) |
| `Jan` … `Dec` | Mean monthly temperature (°C) |

### 1a — Load the Excel file

`pd.read_excel()` reads an Excel file into a pandas **DataFrame**.  
The file has a title row at the top, so the real column headers are on **row index 1** → use `header=1`.

In [ ]:
# Load the data
# pd.read_excel(filename, engine='xlrd', header=row_number)

df = pd.read_excel(???, engine='xlrd', header=???)

print(f'Loaded {len(df)} stations.')

<details><summary>💡 Hint (click to expand)</summary>

```python
df = pd.read_excel('Monthly_Temperature_Update2.xls', engine='xlrd', header=1)
```
</details>

### 1b — Inspect the DataFrame

Before doing any analysis, always look at your data. Run the two cells below — you do not need to change them.

In [ ]:
# Show the first 5 rows
df.head()

In [ ]:
# Show data types and check for missing values
print('Shape:', df.shape)
print()
print(df.dtypes)
print()
print('Missing values per column:')
print(df.isnull().sum())

### 1c — Ensure latitude is numeric

The `Lat_num` column should already be a number, but let's confirm and convert just in case.

In [ ]:
# Convert Lat_num to a numeric type, coercing any bad values to NaN
df['Lat_num'] = pd.to_numeric(df['Lat_num'], errors='coerce')

print('Latitude range: {:.1f}° to {:.1f}°'.format(df['Lat_num'].min(), df['Lat_num'].max()))

> **❓ Quick Check:** What is the approximate latitude range in this dataset? Does that surprise you given the dataset is labeled "Arctic"?

---
## Part 2 — Compute Annual Mean Temperature

Each station has 12 monthly temperature values. We need to average them to get a single **annual mean temperature** for each station.  

The month columns are: `Jan`, `Feb`, `Mar`, `Apr`, `May`, `Jun`, `Jul`, `Aug`, `Sep`, `Oct`, `Nov`, `Dec`.

### 2a — Define the list of month columns

In [ ]:
# Fill in the list of 12 month column names
month_cols = [???]

print('Month columns:', month_cols)

<details><summary>💡 Hint (click to expand)</summary>

```python
month_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
```
</details>

### 2b — Calculate the annual mean temperature

Use `.mean(axis=1)` to average **across columns** (i.e., across months) for each row (station).  
Store the result in a new column called `Annual_Mean`.

In [ ]:
# Calculate the annual mean temperature for each station
# df[month_cols].mean(axis=???) averages across months

df['Annual_Mean'] = df[month_cols].mean(axis=???)

# Preview the result
df[['Station', 'Lat', 'Lat_num', 'Annual_Mean']].head(10)

<details><summary>💡 Hint (click to expand)</summary>

`axis=1` means "average along axis 1", which is the column axis — so it averages across the 12 month columns for each row.

```python
df['Annual_Mean'] = df[month_cols].mean(axis=1)
```
</details>

In [ ]:
# Summary statistics for the annual mean temperature
print(df['Annual_Mean'].describe().round(2))

---
## Part 3 — Figure 1: Annual Mean Temperature vs. Latitude (All Stations)

Now we'll make a **scatter plot** of annual mean temperature vs. latitude. Each dot represents one weather station.

Recall: a **scatter plot** uses `plt.scatter(x, y)`, not `plt.plot(x, y)` (which would connect the dots with a line).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# --- FILL IN: create a scatter plot ---
# x-axis: latitude (Lat_num column)
# y-axis: annual mean temperature (Annual_Mean column)
# Suggested style: s=15 (marker size), alpha=0.5 (transparency), color='steelblue'

ax.scatter(???, ???, s=15, alpha=0.5, color='steelblue')

# --- FILL IN: axis labels and title ---
ax.set_xlabel(???)
ax.set_ylabel(???)
ax.set_title(???)

# Reference line at 0°C
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', label='0°C')
ax.legend(fontsize=9)

ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('fig1_scatter_all_stations.png', dpi=150)
plt.show()
print('Figure 1 saved.')

<details><summary>💡 Hint (click to expand)</summary>

```python
ax.scatter(df['Lat_num'], df['Annual_Mean'], s=15, alpha=0.5, color='steelblue')
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Annual Mean Temperature (°C)')
ax.set_title('Annual Mean Temperature vs. Latitude — All Stations')
```
</details>

---
## Part 4 — Figure 2: Latitude-Binned Average Temperature

Individual stations vary a lot (elevation, proximity to ocean, etc.). To see the **large-scale latitude trend** more clearly, we group stations into **5-degree latitude bins** and plot the average temperature for each bin.

### 4a — Create 5-degree latitude bins

`pd.cut()` divides a continuous variable into discrete intervals.  
We want bins like 20–25°N, 25–30°N, etc., from the minimum to the maximum latitude in the dataset.

In [ ]:
# Create bin edges from the data's latitude range in 5-degree steps
lat_min = np.floor(df['Lat_num'].min() / 5) * 5   # round down to nearest 5
lat_max = np.ceil(df['Lat_num'].max() / 5) * 5    # round up to nearest 5

bin_edges = np.arange(lat_min, lat_max + 5, 5)
print('Bin edges:', bin_edges)

In [ ]:
# Assign each station to a latitude bin
# pd.cut() returns a Categorical column of interval labels

df['Lat_bin'] = pd.cut(df['Lat_num'], bins=bin_edges)

# Preview
df[['Station', 'Lat_num', 'Lat_bin', 'Annual_Mean']].head(8)

### 4b — Compute the mean temperature for each latitude bin

Use `.groupby()` to group stations by their bin, then `.mean()` to average the annual temperature within each group.

In [ ]:
# Group by Lat_bin and compute the mean Annual_Mean for each group
# FILL IN the column to average

binned = df.groupby('Lat_bin', observed=True)[???].mean().reset_index()
binned.columns = ['Lat_bin', 'Bin_Mean_Temp']

# Add a numeric latitude column = midpoint of each bin
binned['Lat_mid'] = binned['Lat_bin'].apply(lambda b: b.mid)

# Show how many stations fall in each bin
bin_counts = df.groupby('Lat_bin', observed=True).size().reset_index(name='N_stations')
binned = binned.merge(bin_counts, on='Lat_bin')

print(binned.to_string(index=False))

<details><summary>💡 Hint (click to expand)</summary>

```python
binned = df.groupby('Lat_bin', observed=True)['Annual_Mean'].mean().reset_index()
```
</details>

### 4c — Plot the binned averages

Make a second figure showing the **binned group averages** vs. the midpoint latitude of each bin.  
You may use a scatter plot or a line plot for this one — choose whichever you think communicates the trend better.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# --- FILL IN: plot the binned averages ---
# x: Lat_mid  |  y: Bin_Mean_Temp
# You can use ax.scatter() or ax.plot() — your choice.
# Consider using the N_stations column to size the markers (e.g., s=binned['N_stations']*3)

ax.???(???, ???)   # replace ??? with your chosen plot command and data

# --- FILL IN: labels and title ---
ax.set_xlabel(???)
ax.set_ylabel(???)
ax.set_title(???)

# Reference line at 0°C
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', label='0°C')
ax.legend(fontsize=9)

ax.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig('fig2_binned_averages.png', dpi=150)
plt.show()
print('Figure 2 saved.')

<details><summary>💡 Hint (click to expand)</summary>

One good option:
```python
ax.scatter(binned['Lat_mid'], binned['Bin_Mean_Temp'],
           s=binned['N_stations']*3, color='tomato', edgecolors='darkred', linewidths=0.5, zorder=3)
ax.plot(binned['Lat_mid'], binned['Bin_Mean_Temp'], color='tomato', linewidth=1.2, zorder=2)
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Mean Temperature (°C)')
ax.set_title('Annual Mean Temperature — 5° Latitude Bin Averages')
```
Marker size proportional to the number of stations in each bin is a great way to show data density!
</details>

---
## Part 5 — Side-by-Side Comparison (Bonus / Optional)

Plotting both figures side by side makes it easy to compare them directly. Run the cell below — no fill-ins needed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# --- Left panel: all stations ---
axes[0].scatter(df['Lat_num'], df['Annual_Mean'], s=12, alpha=0.4, color='steelblue')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Latitude (°N)')
axes[0].set_ylabel('Annual Mean Temperature (°C)')
axes[0].set_title('All Individual Stations')
axes[0].grid(True, linestyle=':', alpha=0.4)

# --- Right panel: binned averages ---
axes[1].scatter(binned['Lat_mid'], binned['Bin_Mean_Temp'],
                s=binned['N_stations']*3, color='tomato',
                edgecolors='darkred', linewidths=0.5, zorder=3)
axes[1].plot(binned['Lat_mid'], binned['Bin_Mean_Temp'],
             color='tomato', linewidth=1.2, zorder=2)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Latitude (°N)')
axes[1].set_title('5° Latitude Bin Averages\n(marker size ∝ # stations)')
axes[1].grid(True, linestyle=':', alpha=0.4)

fig.suptitle('Annual Mean Temperature vs. Latitude', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig3_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison figure saved.')

---
## Part 6 — Discussion Questions

**Answer each question in the cell below it.** Write 2–4 sentences per question.

### Q1 — General trend

Describe the overall relationship between latitude and annual mean temperature seen in Figure 1. Is it what you expected? Why or why not?

**Your answer here:**


### Q2 — Scatter vs. binned

Compare Figure 1 (all stations) and Figure 2 (binned averages). What information is visible in Figure 1 that is hidden in Figure 2? What information is clearer in Figure 2?

**Your answer here:**


### Q3 — Outliers and variability

In Figure 1, you likely see considerable scatter (spread) at any given latitude — stations at the same latitude can have quite different annual mean temperatures. List **at least two physical reasons** why two stations at the same latitude might have different annual mean temperatures.

**Your answer here:**


### Q4 — Station density

Look at the `N_stations` column in the binned table. Are stations evenly distributed across latitudes? Where are they most and least dense? Why might that be?

**Your answer here:**


### Q5 — Climate zone implications

The 0°C line is plotted on both figures. Which latitude range marks the approximate transition from sub-freezing annual means to above-freezing annual means? What climatological or ecological significance does that boundary have?

**Your answer here:**


---
## Submission Checklist

Before uploading to Canvas, make sure you have:

- [ ] Filled in all `???` blanks and run every code cell without errors
- [ ] Figure 1 shows a scatter plot of all individual stations
- [ ] Figure 2 shows the 5° latitude binned averages
- [ ] All five discussion questions are answered
- [ ] Uploaded **this notebook (.ipynb)** and your **Excel file** to the Canvas assignment

---
*Lab developed for ATMS/ENSC Introduction to Meteorology & Climate*